<a href="https://colab.research.google.com/github/yoshiagent/mathModel-colab/blob/main/notebooks/O2_SecondOrder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 二階系統 Second Order System

對照網頁：[數學建模｜二階系統](https://yoshiagent.github.io/mathModel/O2.html)

標準式 $\ddot y + 2\xi\omega_n\dot y + \omega_n^2 y = \omega_n^2 u(t)$，$G(s) = \dfrac{\omega_n^2}{s^2 + 2\xi\omega_n s + \omega_n^2}$。

| 原檔／原講義 | 內容 | 本筆記本 |
|---|---|---|
| `ord2`、`tf`、`damp` | 建立模型、由 m、c、k 求 $\omega_n$、$\xi$ | 第 1 節 |
| Time Response | 齊次解的三種情況（並驗證係數） | 第 2 節 |
| Detail Solution、Specification | 步階響應、$t_r$、$t_p$、$M_p$、$t_s$ | 第 3 節 |
| `StandardOrder2.m`（`ltiview`） | $\omega_n = 1$、$\xi = 0.3$ 的步階響應特性 | 第 4 節 |
| `MpZeta.m`、`TrZeta.m`、`TrZeta1.m` | $M_p$、$t_r$ 與 $\xi$ 的關係 | 第 5 節 |
| `zetaConstant.m`、`wdConstant.m`、比較圖 | $\xi$／$\omega_n$／$\omega_d$／$\xi\omega_n$ 固定時的比較 | 第 6 節 |
| `pzmap`、`sgrid` | $1/(s^2+5s+6)$ 的極點 | 第 7 節 |
| Frequency Response | Bode 圖、共振頻率、共振峰、頻寬 | 第 8 節 |
| Time vs Frequency | $t_s$、$t_p$ 與頻寬的關係 | 第 9 節 |

只用 Colab 內建的 numpy、scipy、sympy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
import sympy as sp
from scipy import signal
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def ds(x, n=600):
    idx = np.linspace(0, len(x) - 1, min(n, len(x))).astype(int)
    return np.round(np.asarray(x)[idx], 5)


def ord2(wn, z):
    """標準二階系統 wn^2/(s^2+2 z wn s+wn^2)"""
    return signal.lti([wn ** 2], [1, 2 * z * wn, wn ** 2])

## 1. 建立模型

原講義：`wn=2; z=0.2; [num,den] = ord2(wn,z); sys=tf(wn^2,den)`。
由質量－彈簧－阻尼器 $m\ddot y + c\dot y + ky = 0$ 可得 $\omega_n = \sqrt{k/m}$、$\xi = \dfrac{c}{2\sqrt{mk}}$；MATLAB 用 `sys=tf(1,[m c k]); [wn, zeta]=damp(sys)`。

In [2]:
sys = ord2(2, 0.2)
print("num =", sys.num, " den =", sys.den)


def damp(den):
    """仿 MATLAB damp：回傳每個極點的 (wn, zeta)"""
    return [(float(abs(p)), float(-p.real / abs(p))) for p in np.roots(den)]


m, c, k = 1.0, 1.0, 2.0                       # 例：MCK 章的參數
for wn, z in damp([m, c, k]):
    print(f"damp(tf(1,[1 1 2])): wn = {wn:.4f}, zeta = {z:.4f}")
print(f"formula: wn = sqrt(k/m) = {np.sqrt(k / m):.4f}, zeta = c/(2 sqrt(mk)) = {c / (2 * np.sqrt(m * k)):.4f}")

num = [4.]  den = [1.  0.8 4. ]
damp(tf(1,[1 1 2])): wn = 1.4142, zeta = 0.3536
damp(tf(1,[1 1 2])): wn = 1.4142, zeta = 0.3536
formula: wn = sqrt(k/m) = 1.4142, zeta = c/(2 sqrt(mk)) = 0.3536


## 2. 齊次解的三種情況 $\ddot y + 2\xi\omega_n\dot y + \omega_n^2 y = 0,\ y(0) = y_0,\ \dot y(0) = v_0$

特徵方程式 $\lambda^2 + 2\xi\omega_n\lambda + \omega_n^2 = 0$，$\lambda = -\xi\omega_n \pm \omega_n\sqrt{\xi^2-1}$。
網頁上的解析解（第 3 種情況的係數已更正為 $c_1 = y_0$、$c_2 = \frac{v_0 + \xi\omega_n y_0}{\omega_d}$）與數值積分比對：

In [3]:
def free_analytic(t, wn, z, y0, v0):
    if z > 1:
        r = np.sqrt(z ** 2 - 1)
        c1 = (v0 + (z + r) * wn * y0) / (2 * wn * r)
        c2 = (-v0 - (z - r) * wn * y0) / (2 * wn * r)
        return c1 * np.exp((-z * wn + wn * r) * t) + c2 * np.exp((-z * wn - wn * r) * t)
    if z == 1:
        return (y0 + (v0 + wn * y0) * t) * np.exp(-wn * t)
    wd = wn * np.sqrt(1 - z ** 2)
    return np.exp(-z * wn * t) * (y0 * np.cos(wd * t) + (v0 + z * wn * y0) / wd * np.sin(wd * t))


def free_original_slide(t, wn, z, y0, v0):
    """原講義第 3 種情況的寫法（c1、c2 對調）"""
    wd = wn * np.sqrt(1 - z ** 2)
    return np.exp(-z * wn * t) * ((v0 + z * wn * y0) / wd * np.cos(wd * t) + y0 * np.sin(wd * t))


wn, y0, v0 = 2.0, 1.0, 0.5
t = np.linspace(0, 8, 801)
fig = go.Figure()
for z in (2.0, 1.0, 0.2):
    num = solve_ivp(lambda tt, x: [x[1], -2 * z * wn * x[1] - wn ** 2 * x[0]], (0, 8), [y0, v0], t_eval=t, rtol=1e-10, atol=1e-12).y[0]
    err = np.max(np.abs(num - free_analytic(t, wn, z, y0, v0)))
    print(f"zeta = {z}: max |analytic - numerical| = {err:.1e}")
    fig.add_scatter(x=ds(t), y=ds(num), name=f"zeta = {z}")
err_slide = np.max(np.abs(free_original_slide(t, wn, 0.2, y0, v0) - free_analytic(t, wn, 0.2, y0, v0)))
print(f"original slide (c1, c2 swapped): y(0) = {free_original_slide(np.array([0.0]), wn, 0.2, y0, v0)[0]:.4f} instead of y0 = {y0}; max error {err_slide:.3f}")
fig.add_scatter(x=ds(t), y=ds(free_original_slide(t, wn, 0.2, y0, v0)), name="slide formula (swapped c1, c2)", line=dict(dash="dot"))
show(fig, "Free response: overdamped / critically damped / underdamped (wn = 2, y0 = 1, v0 = 0.5)", "t (s)", "y")

zeta = 2.0: max |analytic - numerical| = 1.2e-11
zeta = 1.0: max |analytic - numerical| = 6.3e-12
zeta = 0.2: max |analytic - numerical| = 3.2e-11
original slide (c1, c2 swapped): y(0) = 0.4593 instead of y0 = 1.0; max error 0.541


## 3. 單位步階響應與規格

$$y(t) = 1 - \frac{e^{-\xi\omega_n t}}{\sqrt{1-\xi^2}}\sin(\omega_d t + \phi),\qquad \omega_d = \omega_n\sqrt{1-\xi^2},\quad \phi = \cos^{-1}\xi$$

$$t_r = \frac{\pi-\phi}{\omega_d},\quad t_p = \frac{\pi}{\omega_d},\quad M_p = e^{-\xi\pi/\sqrt{1-\xi^2}},\quad t_s \approx \frac{4}{\xi\omega_n}\ (2\%)$$

先用 sympy 驗證 Detail Solution 的推導，再和數值步階響應比對（這裡的 $t_r$ 是 0→100% 第一次到達 1 的時間）。

In [4]:
ts_, zs, wns = sp.symbols("t xi omega_n", positive=True)
wd_s = wns * sp.sqrt(1 - zs ** 2)
ysym = 1 - sp.exp(-zs * wns * ts_) / sp.sqrt(1 - zs ** 2) * sp.sin(wd_s * ts_ + sp.acos(zs))
ode = sp.simplify(ysym.diff(ts_, 2) + 2 * zs * wns * ysym.diff(ts_) + wns ** 2 * ysym - wns ** 2)
print("ODE residual:", ode, "  y(0) =", sp.simplify(ysym.subs(ts_, 0)), "  y'(0) =", sp.simplify(ysym.diff(ts_).subs(ts_, 0)))


def specs(wn, z):
    wd = wn * np.sqrt(1 - z ** 2)
    return dict(tr=(np.pi - np.arccos(z)) / wd, tp=np.pi / wd, Mp=np.exp(-z * np.pi / np.sqrt(1 - z ** 2)), ts=4 / (z * wn))


def measure(wn, z, tf=60, n=60001):
    tt, yy = signal.step(ord2(wn, z), T=np.linspace(0, tf, n))
    ip = np.argmax(yy)
    out = np.where(np.abs(yy - 1) > 0.02)[0]
    return dict(tr100=tt[np.argmax(yy >= 1)], tr1090=tt[np.argmax(yy >= 0.9)] - tt[np.argmax(yy >= 0.1)],
                tp=tt[ip], Mp=yy[ip] - 1, ts=tt[out[-1] + 1]), tt, yy


for wn, z in [(2, 0.2), (1, 0.3), (2, 0.5)]:
    a, (m_, _, _) = specs(wn, z), measure(wn, z)
    print(f"wn={wn}, zeta={z}:  tr {a['tr']:.3f}/{m_['tr100']:.3f}   tp {a['tp']:.3f}/{m_['tp']:.3f}   "
          f"Mp {a['Mp']:.4f}/{m_['Mp']:.4f}   ts(4/zeta wn) {a['ts']:.2f} vs actual 2% {m_['ts']:.2f}   (formula/simulation)")

ODE residual: 0   y(0) = 0   y'(0) = 0
wn=2, zeta=0.2:  tr 0.904/0.905   tp 1.603/1.603   Mp 0.5266/0.5266   ts(4/zeta wn) 10.00 vs actual 2% 9.80   (formula/simulation)


wn=1, zeta=0.3:  tr 1.966/1.967   tp 3.293/3.293   Mp 0.3723/0.3723   ts(4/zeta wn) 13.33 vs actual 2% 11.23   (formula/simulation)
wn=2, zeta=0.5:  tr 1.209/1.210   tp 1.814/1.814   Mp 0.1630/0.1630   ts(4/zeta wn) 4.00 vs actual 2% 4.04   (formula/simulation)


## 4. `StandardOrder2.m`：$\omega_n = 1$、$\xi = 0.3$

```matlab
wn=1; zeta=0.3;
num=[wn^2]; den=[1 2*zeta*wn wn^2];
sys1=tf(num,den);
ltiview(sys1)
```
原講義在 ltiview 勾選 Characteristics，得到 Rise Time 1.33 s（10%→90%）、Peak amplitude 1.37、Overshoot 37.2%、At time 3.31 s、Settling Time 11.2 s（2%）。

In [5]:
m_, tt, yy = measure(1, 0.3)
print(f"rise time (10-90%) = {m_['tr1090']:.2f} s, peak amplitude = {1 + m_['Mp']:.2f}, overshoot = {100 * m_['Mp']:.1f} %, "
      f"at time = {m_['tp']:.2f} s, settling time (2%) = {m_['ts']:.1f} s")
k = tt <= 20
fig = go.Figure()
fig.add_scatter(x=ds(tt[k]), y=ds(yy[k]), name="step")
fig.add_scatter(x=[m_["tp"]], y=[1 + m_["Mp"]], mode="markers+text", text=[f"peak {1 + m_['Mp']:.2f} @ {m_['tp']:.2f} s"], textposition="top right", showlegend=False)
for lv in (0.98, 1.02):
    fig.add_hline(y=lv, line_dash="dot", line_color="gray")
show(fig, "StandardOrder2.m: wn = 1, zeta = 0.3", "t (s)", "Amplitude")

rise time (10-90%) = 1.32 s, peak amplitude = 1.37, overshoot = 37.2 %, at time = 3.29 s, settling time (2%) = 11.2 s


## 5. $M_p$、$t_r$ 與阻尼比（`MpZeta.m`、`TrZeta.m`、`TrZeta1.m`）

```matlab
z=0:0.01:0.7;
Mp=exp(-z*pi./sqrt(1-z.^2));      % MpZeta.m
tr=(pi-acos(z))/2./sqrt(1-z.^2);  % TrZeta.m（wn = 2）
trn=pi-acos(z);                    % TrZeta1.m
```
反過來由 $M_p$ 求 $\xi = \dfrac{-\ln M_p}{\sqrt{\pi^2 + \ln^2 M_p}}$。

In [6]:
z = np.arange(0, 0.7001, 0.01)
Mp = np.exp(-z * np.pi / np.sqrt(1 - z ** 2))
fig = make_subplots(rows=1, cols=3, subplot_titles=("MpZeta.m: Mp", "TrZeta.m: tr (wn = 2)", "TrZeta1.m: pi - acos(zeta)"))
fig.add_scatter(x=z, y=np.round(Mp, 5), name="Mp", row=1, col=1)
fig.add_scatter(x=z, y=np.round((np.pi - np.arccos(z)) / 2 / np.sqrt(1 - z ** 2), 5), name="tr", row=1, col=2)
fig.add_scatter(x=z, y=np.round(np.pi - np.arccos(z), 5), name="pi-acos", row=1, col=3)
fig.update_xaxes(title="zeta")
show(fig, "Overshoot and rise time vs damping ratio", h=380)
for mp in (0.37, 0.163, 0.05):
    print(f"Mp = {mp}: zeta = {-np.log(mp) / np.sqrt(np.pi ** 2 + np.log(mp) ** 2):.3f}")

Mp = 0.37: zeta = 0.302
Mp = 0.163: zeta = 0.500
Mp = 0.05: zeta = 0.690


## 6. 固定一個參數的比較

* `zetaConstant.m`：$\xi = 0.5$，$\omega_n$ = 6、3、1 → 超越量相同（0.163），$t_p = 3.63/\omega_n$。
* $\omega_n = 2$，$\xi$ = 0.3、0.5、0.7 → $M_p$ = 0.37、0.16、0.05。
* `wdConstant.m`：$\omega_d = 3$，$\xi\omega_n$ = 0.5、1、2 → 振盪頻率與 $t_p = \pi/3 = 1.05$ 相同，衰減快慢不同。
* $\xi\omega_n = 1$，$\omega_d$ = 10、5、3 → 包絡線相同，$t_s \approx 4$。

In [7]:
def from_wd_a(wd, a):
    """wdConstant.m 的換算：給 wd 與 a = zeta*wn，求 zeta、wn"""
    z = 1 / np.sqrt(1 + (wd / a) ** 2)
    return a / z, z


cases = {
    "zeta = 0.5, wn = 6 / 3 / 1": [(6, 0.5), (3, 0.5), (1, 0.5)],
    "wn = 2, zeta = 0.3 / 0.5 / 0.7": [(2, 0.3), (2, 0.5), (2, 0.7)],
    "wd = 3, zeta*wn = 0.5 / 1 / 2": [from_wd_a(3, a) for a in (0.5, 1, 2)],
    "zeta*wn = 1, wd = 10 / 5 / 3": [from_wd_a(wd, 1) for wd in (10, 5, 3)],
}
fig = make_subplots(rows=2, cols=2, subplot_titles=list(cases))
for i, (name, lst) in enumerate(cases.items()):
    r, c = divmod(i, 2)
    for wn, zz in lst:
        m_, tt, yy = measure(wn, zz, tf=12, n=6001)
        fig.add_scatter(x=ds(tt, 500), y=ds(yy, 500), name=f"wn={wn:.2f}, zeta={zz:.3f}", row=r + 1, col=c + 1)
        print(f"{name:32s} wn={wn:6.3f} zeta={zz:.3f}: tp={m_['tp']:.3f}  Mp={m_['Mp']:.3f}  ts(2%)={m_['ts']:.2f}  wd={wn * np.sqrt(1 - zz ** 2):.3f}")
show(fig, "Step responses with one parameter held constant", h=700)

zeta = 0.5, wn = 6 / 3 / 1       wn= 6.000 zeta=0.500: tp=0.604  Mp=0.163  ts(2%)=1.35  wd=5.196
zeta = 0.5, wn = 6 / 3 / 1       wn= 3.000 zeta=0.500: tp=1.210  Mp=0.163  ts(2%)=2.69  wd=2.598
zeta = 0.5, wn = 6 / 3 / 1       wn= 1.000 zeta=0.500: tp=3.628  Mp=0.163  ts(2%)=8.08  wd=0.866
wn = 2, zeta = 0.3 / 0.5 / 0.7   wn= 2.000 zeta=0.300: tp=1.646  Mp=0.372  ts(2%)=5.62  wd=1.908
wn = 2, zeta = 0.3 / 0.5 / 0.7   wn= 2.000 zeta=0.500: tp=1.814  Mp=0.163  ts(2%)=4.04  wd=1.732
wn = 2, zeta = 0.3 / 0.5 / 0.7   wn= 2.000 zeta=0.700: tp=2.200  Mp=0.046  ts(2%)=2.99  wd=1.428
wd = 3, zeta*wn = 0.5 / 1 / 2    wn= 3.041 zeta=0.164: tp=1.048  Mp=0.592  ts(2%)=7.56  wd=3.000


wd = 3, zeta*wn = 0.5 / 1 / 2    wn= 3.162 zeta=0.316: tp=1.048  Mp=0.351  ts(2%)=3.54  wd=3.000
wd = 3, zeta*wn = 0.5 / 1 / 2    wn= 3.606 zeta=0.555: tp=1.048  Mp=0.123  ts(2%)=1.62  wd=3.000
zeta*wn = 1, wd = 10 / 5 / 3     wn=10.050 zeta=0.100: tp=0.314  Mp=0.730  ts(2%)=3.82  wd=10.000
zeta*wn = 1, wd = 10 / 5 / 3     wn= 5.099 zeta=0.196: tp=0.628  Mp=0.533  ts(2%)=3.88  wd=5.000
zeta*wn = 1, wd = 10 / 5 / 3     wn= 3.162 zeta=0.316: tp=1.048  Mp=0.351  ts(2%)=3.54  wd=3.000


## 7. 極點圖（`pzmap`、`sgrid`）

原講義：`sys=tf(1,[1 5 6]); pzmap(sys); sgrid`。$s^2 + 5s + 6 = (s+2)(s+3)$，兩個實數極點（過阻尼）。

In [8]:
p = np.roots([1, 5, 6])
print("poles:", p, " damp:", [(round(w, 4), round(zz, 4)) for w, zz in damp([1, 5, 6])])
wn6, z6 = np.sqrt(6), 5 / (2 * np.sqrt(6))
print(f"as standard form: wn = sqrt(6) = {wn6:.4f}, zeta = {z6:.4f} (> 1, overdamped); DC gain = 1/6 = {1 / 6:.4f}")
fig = go.Figure()
th = np.linspace(np.pi / 2, 3 * np.pi / 2, 100)
for r in (1, 2, 3):                                        # sgrid：等 wn 半圓
    fig.add_scatter(x=np.round(r * np.cos(th), 4), y=np.round(r * np.sin(th), 4), mode="lines", line=dict(color="lightgray", dash="dot"), showlegend=False)
for zz in (0.3, 0.52, 0.7, 0.82, 0.9, 0.95):               # sgrid：等 zeta 射線
    s_ = np.sqrt(1 - zz ** 2)                               # 極點 -zeta*r ± j*r*sqrt(1-zeta^2)
    fig.add_scatter(x=[0, -4 * zz, None, 0, -4 * zz], y=[0, 4 * s_, None, 0, -4 * s_], mode="lines", line=dict(color="lightgray", dash="dot"), showlegend=False)
fig.add_scatter(x=p.real, y=p.imag, mode="markers", marker=dict(symbol="x", size=12, color="blue"), name="poles")
fig.update_xaxes(range=[-3.5, 0.2])
fig.update_yaxes(range=[-1, 1])
show(fig, "pzmap(tf(1,[1 5 6])) with sgrid", "Real Axis", "Imaginary Axis", h=380)

poles: [-3. -2.]  damp: [(3.0, 1.0), (2.0, 1.0)]
as standard form: wn = sqrt(6) = 2.4495, zeta = 1.0206 (> 1, overdamped); DC gain = 1/6 = 0.1667


## 8. 頻率響應：Bode 圖、共振與頻寬

$$M(u) = \frac{1}{\sqrt{(1-u^2)^2 + 4\xi^2u^2}},\quad u = \frac{\omega}{\omega_n}$$

$$u_r = \sqrt{1-2\xi^2}\ (\xi < 0.707),\qquad M_{p\omega} = \frac{1}{2\xi\sqrt{1-\xi^2}},\qquad \frac{\omega_{BW}}{\omega_n} = \sqrt{(1-2\xi^2) + \sqrt{(1-2\xi^2)^2 + 1}}$$

In [9]:
u = np.logspace(-1, 1, 2001)
fig = make_subplots(rows=1, cols=2, subplot_titles=("20 log M(u) (dB)", "Phase (deg)"))
for zz in (0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0):
    G = 1 / ((1 - u ** 2) + 2j * zz * u)
    fig.add_scatter(x=ds(u, 400), y=ds(20 * np.log10(abs(G)), 400), name=f"zeta={zz}", legendgroup=str(zz), row=1, col=1)
    fig.add_scatter(x=ds(u, 400), y=ds(np.degrees(np.angle(G)), 400), showlegend=False, legendgroup=str(zz), row=1, col=2)
fig.update_xaxes(type="log", title="u = w/wn")
show(fig, "Second-order system: Bode diagram", h=460)

for zz in (0.2, 0.3, 0.5, 0.707):
    G = 1 / ((1 - u ** 2) + 2j * zz * u)
    M = abs(G)
    Mf = lambda x: 1 / np.sqrt((1 - x ** 2) ** 2 + 4 * zz ** 2 * x ** 2)
    ubw_num = brentq(lambda x: Mf(x) - 1 / np.sqrt(2), 0.01, 10)          # 數值求 -3 dB 點
    ubw = np.sqrt((1 - 2 * zz ** 2) + np.sqrt((1 - 2 * zz ** 2) ** 2 + 1))
    if zz < 0.707:                                                     # zeta >= 0.707（1/sqrt2）沒有共振峰
        r = minimize_scalar(lambda x: -Mf(x), bounds=(0.01, 2), method="bounded", options=dict(xatol=1e-10))
        ur, Mpw = np.sqrt(1 - 2 * zz ** 2), 1 / (2 * zz * np.sqrt(1 - zz ** 2))
        res = f"ur {ur:.4f}/{r.x:.4f}   Mpw {Mpw:.4f}/{-r.fun:.4f}"
    else:
        res = "no resonant peak (zeta >= 0.707)"
    print(f"zeta={zz}: {res}   wBW/wn {ubw:.4f}/{ubw_num:.4f}   linear approx {-1.19 * zz + 1.85:.3f}   (formula/numerical)")

zeta=0.2: ur 0.9592/0.9592   Mpw 2.5516/2.5516   wBW/wn 1.5096/1.5096   linear approx 1.612   (formula/numerical)
zeta=0.3: ur 0.9055/0.9055   Mpw 1.7471/1.7471   wBW/wn 1.4537/1.4537   linear approx 1.493   (formula/numerical)
zeta=0.5: ur 0.7071/0.7071   Mpw 1.1547/1.1547   wBW/wn 1.2720/1.2720   linear approx 1.255   (formula/numerical)
zeta=0.707: no resonant peak (zeta >= 0.707)   wBW/wn 1.0002/1.0002   linear approx 1.009   (formula/numerical)


相位：$u = 0$ 時 0°，$u = 1$ 時 $\angle(-j2\xi) = -90°$，$u = 10$ 時 $\angle(-99 - j20\xi) \to -180°$。

In [10]:
for zz in (0.1, 0.5):
    print(f"zeta={zz}: phase at u=1: {np.degrees(np.angle(1 / (2j * zz))):.1f} deg, at u=10: {np.degrees(np.angle(1 / ((1 - 100) + 2j * zz * 10))):.2f} deg")

zeta=0.1: phase at u=1: -90.0 deg, at u=10: -178.84 deg
zeta=0.5: phase at u=1: -90.0 deg, at u=10: -174.23 deg


## 9. 時間響應與頻寬的關係

由 $t_s = 4/(\xi\omega_n)$ 或 $t_p = \pi/(\omega_n\sqrt{1-\xi^2})$ 先求 $\omega_n$，再代入頻寬公式：
$$\omega_{BW} = \frac{4}{t_s\xi}\sqrt{(1-2\xi^2) + \sqrt{(1-2\xi^2)^2 + 1}} = \frac{\pi}{t_p\sqrt{1-\xi^2}}\sqrt{(1-2\xi^2) + \sqrt{(1-2\xi^2)^2 + 1}}$$
以 $\omega_n = 2$、$\xi = 0.5$ 驗證：兩條公式算出的頻寬一致，也等於數值 Bode 圖量到的 −3 dB 頻率。

In [11]:
wn, zz = 2.0, 0.5
sp_ = specs(wn, zz)
root = np.sqrt((1 - 2 * zz ** 2) + np.sqrt((1 - 2 * zz ** 2) ** 2 + 1))
w = np.logspace(-1, 1.5, 20001)
_, mag, _ = signal.bode(ord2(wn, zz), w)
print(f"from ts: {4 / (sp_['ts'] * zz) * root:.4f}   from tp: {np.pi / (sp_['tp'] * np.sqrt(1 - zz ** 2)) * root:.4f}   "
      f"numerical -3 dB: {brentq(lambda x: abs(wn ** 2 / ((1j * x) ** 2 + 2 * zz * wn * 1j * x + wn ** 2)) - 1 / np.sqrt(2), 0.1, 20):.4f} rad/s")

from ts: 2.5440   from tp: 2.5440   numerical -3 dB: 2.5440 rad/s
